<a href="https://colab.research.google.com/github/sebarom06/econ3916-statsml/blob/main/Assignment3/Econ_3916_Assignment_3_Causal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Step 1.1: The Zero-Inflated Gig Economy Tip Distribution
np.random.seed(42)
zeros = np.zeros(100)
tips = np.random.exponential(scale=5.0, size=150)
driver_tips = np.concatenate([zeros, tips])

# Step 1.2: The Manual Bootstrap Engine
n_bootstrap = 10000
boot_medians = np.array([
    np.median(np.random.choice(driver_tips, size=len(driver_tips), replace=True))
    for _ in range(n_bootstrap)
])

ci_lower, ci_upper = np.percentile(boot_medians, [2.5, 97.5])

print(f"Observed Median:      ${np.median(driver_tips):.4f}")
print(f"Bootstrap 95% CI:     [${ci_lower:.4f}, ${ci_upper:.4f}]")
print(f"Lower margin:         ${np.median(driver_tips) - ci_lower:.4f}")
print(f"Upper margin:         ${ci_upper - np.median(driver_tips):.4f}")

# Step 2.1: The Algorithmic Routing Crash
np.random.seed(42)
control   = np.random.normal(loc=35, scale=5, size=500)
treatment = np.random.lognormal(mean=3.4, sigma=0.4, size=500)

observed_diff = control.mean() - treatment.mean()

print(f"Control Mean:         {control.mean():.4f} min")
print(f"Treatment Mean:       {treatment.mean():.4f} min")
print(f"Observed Difference:  {observed_diff:.4f} min")

# Step 2.2: The Exact Non-Parametric Permutation Test
combined = np.concatenate([control, treatment])

n_permutations = 5000
perm_diffs = np.array([
    np.random.permutation(combined)[:500].mean() - np.random.permutation(combined)[500:].mean()
    for _ in range(n_permutations)
])

p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

print(f"\nObserved Difference:  {observed_diff:.4f}")
print(f"Permutation P-value:  {p_value:.4f}")
print(f"Significant (α=0.05): {p_value < 0.05}")

# Step 3.1:
from google.colab import files
import io

uploaded = files.upload()  # upload swiftcart_loyalty.csv when prompted
df = pd.read_csv(io.BytesIO(uploaded['swiftcart_loyalty.csv']))

subscribers     = df[df['subscriber'] == 1]['post_spend']
non_subscribers = df[df['subscriber'] == 0]['post_spend']

sdo = subscribers.mean() - non_subscribers.mean()

print(f"Subscriber Mean Post-Spend:     ${subscribers.mean():.4f}")
print(f"Non-Subscriber Mean Post-Spend: ${non_subscribers.mean():.4f}")
print(f"Naive SDO (D=1 minus D=0):      ${sdo:.4f}")

# Step 3.2
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

covariates = ['pre_spend', 'account_age', 'support_tickets']
X = df[covariates].values
D = df['subscriber'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_scaled, D)
df['propensity_score'] = logit.predict_proba(X_scaled)[:, 1]

treated = df[df['subscriber'] == 1].copy()
control = df[df['subscriber'] == 0].copy()

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(control[['propensity_score']].values)

distances, indices = nn.kneighbors(treated[['propensity_score']].values)
matched_control = control.iloc[indices.flatten()].copy()

att = treated['post_spend'].values.mean() - matched_control['post_spend'].values.mean()

print(f"\nNaive SDO:                   ${sdo:.4f}")
print(f"Causal ATT (PSM):            ${att:.4f}")
print(f"Selection Bias Component:    ${sdo - att:.4f}")

# Task 4.1
covariates = ['pre_spend', 'account_age', 'support_tickets']

# Build unmatched dataframe
df_unmatched = df.copy()

# Build matched dataframe
df_matched = pd.concat([
    treated.assign(matched_group='treated'),
    matched_control.assign(matched_group='control')
])

def standardized_mean_diff(df, covariate):
    treated_vals = df[df['subscriber'] == 1][covariate]
    control_vals = df[df['subscriber'] == 0][covariate]
    pooled_std   = np.sqrt((treated_vals.std()**2 + control_vals.std()**2) / 2)
    return (treated_vals.mean() - control_vals.mean()) / pooled_std

smd_before = [standardized_mean_diff(df_unmatched, c) for c in covariates]
smd_after  = [standardized_mean_diff(df_matched,   c) for c in covariates]

smd_df = pd.DataFrame({
    'Covariate': covariates,
    'Before Matching': smd_before,
    'After Matching':  smd_after
})

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(smd_df['Before Matching'], smd_df['Covariate'],
           color='crimson', s=100, zorder=5, label='Before Matching')
ax.scatter(smd_df['After Matching'],  smd_df['Covariate'],
           color='steelblue', s=100, zorder=5, label='After Matching')

for _, row in smd_df.iterrows():
    ax.plot([row['Before Matching'], row['After Matching']],
            [row['Covariate'], row['Covariate']],
            color='grey', linewidth=1.2, linestyle='--', zorder=3)

ax.axvline(0,     color='black',      linewidth=1.2, linestyle='-')
ax.axvline( 0.1,  color='green',      linewidth=1,   linestyle=':', alpha=0.7, label='±0.1 Balance Threshold')
ax.axvline(-0.1,  color='green',      linewidth=1,   linestyle=':', alpha=0.7)

ax.set_xlabel('Standardized Mean Difference (SMD)', fontsize=11)
ax.set_title('Love Plot — Covariate Balance Before & After PSM', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

print("\nSMD Table:")
print(smd_df.to_string(index=False))

Observed Median:      $0.7553
Bootstrap 95% CI:     [$0.2653, $1.3636]
Lower margin:         $0.4900
Upper margin:         $0.6082
Control Mean:         35.0342 min
Treatment Mean:       32.7692 min
Observed Difference:  2.2650 min

Observed Difference:  2.2650
Permutation P-value:  0.0000
Significant (α=0.05): True
